# Topic 07 · MOSS-TTS: from raw audio to a speech generator: rebuilding the technical report's small steps

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd07-moss-tts-en.ipynb)

This notebook accompanies [Topic 07 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/moss-tts/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd07-moss-tts.ipynb)): the code,
the data and every check are the same.

Topic 05 rebuilds an audio *tokenizer*. But a text-to-speech (TTS) model also needs **data**: millions of hours of
(audio, transcript) pairs, each pair exactly one speaker, and a **generator** that can write those 32 layers of
tokens. The [*MOSS-TTS*](https://arxiv.org/abs/2603.18090) technical report (SII-OpenMOSS, 2026) tells the whole
pipeline: four stages that filter raw audio from the web, synthetic data for what the web lacks, the
MOSS-Audio-Tokenizer, how 33 token channels are laid out for an autoregressive Transformer, and four phases of
pretraining.

We have neither millions of hours of audio nor an 8-billion-parameter model. But most of the pipeline is made of
**small computations, written out in the report**: a gain formula, three transcript rules, a cosine-based choice of
voice prompt, a shift of the token layers. We write each of them ourselves and run it on real data: the 3,000 spoken
English digits of Topic 05, stitched into a "conversation" between three people.

**What you'll do:**
1. Normalize loudness with stage 1's formula, and see that one sentence of the report has two readings with very
   different results.
2. Build a long multi-speaker recording, write a small speaker diarizer yourself (energy-based speech detection,
   clustering of voice embeddings), then merge segments with stage 2's rules.
3. Transcribe with a digit "judge", run stage 3's three rules and single-speaker check, then stage 4's character
   rate.
4. Choose a voice prompt for each segment: 5 crops cut from every other segment of the same speaker, scored by the
   cosine of voice embeddings.
5. Lay out 33 token channels in the delay pattern, check that it can be undone, count steps against the Local
   Transformer, then run the four-phase learning-rate schedule on a small problem.

**Background:** [Lesson 14 · Tokens & embeddings](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-embedding/),
[Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/),
[Lesson 22 · Pretraining & fine-tuning](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/tien-huan-luyen/) and
[Topic 05 · Audio tokenization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-hoa-am-thanh/).
**Run time:** under a minute on a CPU, including the first data download.

The notebook uses **PyTorch**, like Topic 05. It borrows two pretrained "judges" from that topic (a small CNN that
recognizes digits and a CNN that recognizes voices) and a few real token sequences; every MOSS-TTS step we write and
run ourselves. Every number from the report carries its page number (arXiv v2); every other number is computed by
the cells below.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

In [ ]:
import base64
import hashlib
import io
import json
import math
import os
import re
import tarfile
import urllib.request
import wave
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def num(x, digits=2):
    """A number with thousands separators: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


def sci(x, digits=1):
    """A very small or very large number: 2·10⁻⁴ (0 prints as 0; no trailing “.0”)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)) + 1e-9)
    m = num(x / 10**e, digits).removesuffix(".0")
    return f"{m}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.001 <= v <= 10000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def play(x, label=""):
    """Play an 8 kHz clip in the notebook (loudness kept, clipped to [−1, 1])."""
    if label:
        print(label)
    display(Audio(np.clip(np.asarray(x, np.float64), -1, 1), rate=SR, normalize=False))

## 2 · Data: 3,000 spoken digits, and Topic 05's two judges

The [Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD, version 1.0.10,
CC BY-SA 4.0 license): 6 speakers, each saying each digit from 0 to 9 fifty times, recorded at 8,000 samples per
second, 16 bits per sample. As in Topic 05, we check the MD5 of the *contents* of the 3,000 wav files. Unlike Topic
05, this time we **keep each recording's original loudness**, because stage 1 of the report is precisely the
loudness step.

Topic 05's `codec-weights.json` holds the (int8) weights of two small CNNs, trained on the recordings numbered 10
and up: one guesses the digit, the other guesses the speaker. Here they play the report's speech recognizer (ASR)
and its voice-embedding model. The file also holds a few real 32-layer token sequences, used in section 10.

In [ ]:
SR = 8000
SPEAKERS = ("george", "jackson", "lucas", "nicolas", "theo", "yweweler")
WORDS = ("zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine")
FSDD = "https://github.com/Jakobovski/free-spoken-digit-dataset/archive/refs/tags/v1.0.10.tar.gz"
TGZ = Path("fsdd-v1.0.10.tar.gz")
if not TGZ.exists():
    urllib.request.urlretrieve(FSDD, TGZ)


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


W = fetch_json("codec-weights.json", "6f4b0e5508a869f770f628dd0468e742")

In [ ]:
raw = {}
with tarfile.open(TGZ) as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".wav") and "/recordings/" in m.name:
            raw[m.name.rsplit("/", 1)[1]] = tf.extractfile(m).read()
h = hashlib.md5()
for k in sorted(raw):
    h.update(raw[k])
assert len(raw) == 3000 and h.hexdigest() == "2c93442f2c808e24d4d913f6be24d32c", "FSDD is corrupted; download it again"

waves, digit, spk, idx = [], [], [], []
for name in sorted(raw):
    d, s, i = name[:-4].split("_")
    with wave.open(io.BytesIO(raw[name])) as w:
        assert (w.getnchannels(), w.getsampwidth(), w.getframerate()) == (1, 2, SR)
        waves.append(np.frombuffer(w.readframes(w.getnframes()), "<i2").astype(np.float64) / 32768)
    digit.append(int(d))
    spk.append(SPEAKERS.index(s))
    idx.append(int(i))
digit, spk, idx = np.array(digit), np.array(spk), np.array(idx)
dur = np.array([len(x) for x in waves]) / SR
print(
    f"{num(len(waves), 0)} recordings, {num(dur.sum() / 60, 1)} minutes; {num(dur.min(), 2)}–{num(dur.max(), 2)} s long"
)

In [ ]:
def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, np.float64) / 700)


def mel_filters(n_fft, n_mels):
    """Triangular filters on the mel scale, (n_fft/2 + 1, n_mels), as in Topic 05."""
    freqs = np.linspace(0, SR / 2, n_fft // 2 + 1)
    edges = 700 * (10 ** (np.linspace(hz_to_mel(0), hz_to_mel(SR / 2), n_mels + 2) / 2595) - 1)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    tri = np.maximum(0, np.minimum((freqs[None] - lo) / (mid - lo), (hi - freqs[None]) / (hi - mid)))
    return torch.tensor((tri * (2 / (hi - lo))).T, dtype=torch.float32)


HANN, FB40 = torch.hann_window(256), mel_filters(256, 40)


def logmel(x):
    """(B, L) → (B, 40, frames): log₁₀ magnitude in 40 mel bands; 32 ms window, 10 ms hop."""
    s = torch.stft(x, 256, 80, window=HANN, center=True, return_complex=True).abs()
    return (s.transpose(1, 2) @ FB40).clamp_min(1e-5).log10().transpose(1, 2)


class Judge(nn.Module):
    """Topic 05's judge: a 1-D CNN over 40 log-mel bands → a 128-dimensional vector → classes."""

    def __init__(self, classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(40, 128, 5, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
            nn.Conv1d(128, 128, 5, stride=2, padding=2), nn.ReLU(),
        )  # fmt: skip
        self.emb, self.head = nn.Linear(256, 128), nn.Linear(128, classes)

    def embed(self, x):
        h = self.conv((logmel(x) + 5) / 2)
        return F.relu(self.emb(torch.cat([h.mean(-1), h.amax(-1)], 1)))

    def forward(self, x):
        return self.head(self.embed(x))


def unpack(p):
    """Unpack the weights: int8 × a float16 scale per row, or plain float16."""
    sd = {}
    for k, v in p.items():
        if "q" in v:
            q = np.frombuffer(base64.b64decode(v["q"]), np.int8).astype(np.float32).reshape(v["shape"][0], -1)
            s = np.frombuffer(base64.b64decode(v["s"]), "<f2").astype(np.float32)
            sd[k] = torch.tensor((q * s[:, None]).reshape(v["shape"]))
        else:
            sd[k] = torch.tensor(
                np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float32).reshape(v["shape"])
            )
    return sd


asr, voice = Judge(10), Judge(len(SPEAKERS))
asr.load_state_dict(unpack(W["judgeDigit"]))
voice.load_state_dict(unpack(W["judgeSpk"]))
asr.eval(), voice.eval()


def judge_input(xs):
    """As when the two judges were trained: each clip brought to an RMS of 0.1, zero-padded to at least 1.28 s."""
    n = max(10240, max(len(x) for x in xs))
    out = np.zeros((len(xs), n), np.float32)
    for b, x in enumerate(xs):
        out[b, : len(x)] = x * 0.1 / np.sqrt(np.mean(x**2) + 1e-12)
    return torch.tensor(np.clip(out, -1, 1))


TEST = np.where(idx < 5)[0]  # recordings the two judges have never seen
with torch.no_grad():
    Xt = judge_input([waves[c] for c in TEST])
    acc_asr = float((asr(Xt).argmax(-1).numpy() == digit[TEST]).mean())
    acc_voice = float((voice(Xt).argmax(-1).numpy() == spk[TEST]).mean())
print(f"on {len(TEST)} test recordings: right digit {pct(acc_asr)}, right speaker {pct(acc_voice)}")

## 3 · What the report does, and what we rebuild

The report splits the data pipeline into three parts (section 5.1, pp. 10–14), followed by the tokenizer and the
model:

| Step | Report (page) | Here |
|---|---|---|
| Stage 1 · cleaning | MossFormer2-SE-48K denoising, FLAC, gain clip(−20 − L, −3, 3) dB then division by max \|x\| (p. 11) | the gain formula, under two readings (section 4) |
| Stage 2 · speakers | DiariZen diarization; drop pieces < 0.1 s, merge adjacent same-label segments, at most 1 hour (pp. 11–12) | energy-based speech detection, clustering of voice embeddings, the exact merge rules (section 5) |
| Stage 3 · transcripts | MOSS-Transcribe-Diarize transcription; 3 rules, an LLM that diagnoses and repairs, an [S1]-only check (pp. 12–13) | the digit judge transcribes; 3 rules, the easy part of the LLM, the [S1] check (section 6) |
| Stage 4 · overall | DNSMOS > 2.8, AudioBox PQ > 6.5, Whisper and an LLM agree on the language, character rate (p. 13) | character rate (section 7) |
| Synthesis | voice prompts: 5 crops from every other segment, cosine of WavLM voice embeddings; noisy text, phonetic text; target token count (pp. 13–14) | voice prompts and target token count (section 8) |
| Tokenizer | MOSS-Audio-Tokenizer: 24 kHz → 12.5 frames/s × 32 layers (pp. 4–7) | counting (section 9); Topic 05 rebuilds it |
| Model | 33 channels; delay pattern (8 B) or Local Transformer (1.7 B); loss weighted by λ (pp. 7–10) | laying out real tokens, counting steps, the loss (section 10) |
| Pretraining | four phases, warmup–stable–decay, 2·10⁻⁴ → 2·10⁻⁶ (pp. 15–16) | the same schedule shape on a small problem (section 11) |

What needs large models (denoising, quality scoring, the LLM, Whisper) we do not run; the notebook says so at each
such point.

## 4 · Stage 1: loudness normalization

Audio from the web differs in loudness by tens of dB. The report measures the **RMS level** of the whole recording
in dBFS, decibels relative to the ±1 full scale: a full-scale square wave sits at 0 dBFS, a full-scale sine at about
−3 dBFS, and the quieter a signal, the more negative its level. It then multiplies the recording by a clamped gain
(p. 11):

$$L = 20\log_{10}\sqrt{\tfrac{1}{T}\textstyle\sum_t x_t^2 + \epsilon}, \qquad g = \mathrm{clip}(-20 - L,\ -3,\ 3)\ \text{dB}, \qquad x \leftarrow x \cdot 10^{g/20}$$

The target is −20 dBFS, but the gain never exceeds ±3 dB: a very quiet recording is raised by only 3 dB rather than
pulled straight up to −20, and a loud one is lowered by only 3 dB. Then comes a second step: "divide by the maximum
absolute sample value, to bring the result into [−1, 1]".

In [ ]:
EPS = 1e-10  # ε inside the square root; the report gives no value


def level_dbfs(x):
    return 20 * np.log10(np.sqrt(np.mean(np.square(x)) + EPS))


def gain_db(L, target=-20.0, clamp=3.0):
    return np.clip(target - L, -clamp, clamp)


for L in (-32, -21, -14):
    print(f"L = {num(L, 0)} dBFS → would need {num(-20 - L, 0):>3s} dB, gets {num(gain_db(L), 0)} dB")
assert [gain_db(L) for L in (-32, -21, -14)] == [3, 1, -3]

L0 = np.array([level_dbfs(x) for x in waves])
print(
    f"\nFSDD as recorded: RMS level from {num(L0.min(), 1)} to {num(L0.max(), 1)} dBFS, median {num(np.median(L0), 1)}"
)
for s, name in enumerate(SPEAKERS):
    print(f"  {name:9s} median {num(np.median(L0[spk == s]), 1):>6s} dBFS")
print(f"{pct(np.mean(np.abs(-20 - L0) > 3))} of recordings need more than ±3 dB, so their gain is clamped")

FSDD's six speakers were recorded at very different levels: theo and yweweler are about 12–22 dB quieter than the
other four. Most recordings need more than 3 dB, so the gain is clamped for nearly all of them.

The second step has **two readings**. Read literally (A): *always* divide by max |x|. But then x · 10^{g/20} divided
by max |x · 10^{g/20}| is exactly x / max |x|: the first step's gain cancels out completely, and the final RMS level
depends only on the shape of the wave (its peak-to-RMS ratio). The second reading (B): divide only when some sample
exceeds ±1, i.e. a guard against clipping. The report does not say which.

### 🤔 Predict first
The gap between FSDD's loudest and quietest recordings is about 38 dB. After stage 1, under reading B, how much is
left? And under reading A?

In [ ]:
def stage1(x, always_peak):
    """Stage 1: clamped gain, then division by max|x| (A: always; B: only when it exceeds ±1)."""
    y = x * 10 ** (gain_db(level_dbfs(x)) / 20)
    peak = np.abs(y).max()
    return y / peak if always_peak or peak > 1 else y


ex = waves[0]
assert np.allclose(stage1(ex, True), ex / np.abs(ex).max()), "reading A: the gain cancels out"
LA = np.array([level_dbfs(stage1(x, True)) for x in waves])
LB = np.array([level_dbfs(stage1(x, False)) for x in waves])
n_over = sum(np.abs(x * 10 ** (gain_db(level_dbfs(x)) / 20)).max() > 1 for x in waves)
for name, v in (("original", L0), ("reading B", LB), ("reading A", LA)):
    lo, hi = np.percentile(v, [5, 95])
    print(f"{name:10s} loudest − quietest {num(v.max() - v.min(), 1):>5s} dB · "
          f"5%–95%: {num(lo, 1)} … {num(hi, 1)} dBFS")  # fmt: skip
print(f"reading B divides only {num(n_over, 0)} recordings (some sample above ±1 after the gain)")

The figure below plots each speaker's RMS levels (each box is 500 recordings): as recorded, after reading B and after
reading A.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, (title, v, col, hatch) in zip(
    axes,
    (
        ("as recorded", L0, GREY, ""),
        ("reading B: clipping guard only", LB, BLUE, "//"),
        ("reading A: always divide by max|x|", LA, ORANGE, ".."),
    ),
    strict=True,
):
    bp = ax.boxplot([v[spk == s] for s in range(6)], patch_artist=True, widths=0.6, showfliers=False)
    for b in bp["boxes"]:
        b.set(facecolor=col, alpha=0.55, hatch=hatch)
    ax.axhline(-20, color="k", lw=0.8, ls="--")
    ax.set_xticks(range(1, 7), SPEAKERS, rotation=45, fontsize=8)
    ax.set_title(title, fontsize=10)
axes[0].set_ylabel("RMS level L (dBFS)")
axes[0].text(6.45, -21, "target −20", ha="right", va="top", fontsize=8)
fmt_axes(fig)
plt.tight_layout()
plt.show()

Under reading B, the gap only shrinks from 37.9 to 31.9 dB: ±3 dB is far too little against the 12–22 dB between
speakers, so theo and yweweler stay much quieter. Under reading A, every recording peaks at 1, and the RMS levels
crowd into a much narrower band (a gap of 16.8 dB); but then the ±3 dB gain formula is redundant. The report says
the ±3 dB range is there to "avoid over-compressing already quiet recordings and over-amplifying abnormal ones", a
purpose that only makes sense under reading B. The code for this step is not published, so we cannot be sure.

The general lesson: a pipeline described in words can hide an important choice. To reproduce it, turn each sentence
into code, then check whether it cancels itself out.

## 5 · Stage 2: diarization, then merging

A recording from the web (a podcast episode, a film) has several speakers. A *speaker diarization* system returns a
list of time spans, each with a speaker label, and **a label only means something within that recording**:
SPEAKER_00 of one episode is not SPEAKER_00 of another (p. 11). The report uses DiariZen. We build a small one on the
same idea: detect the spans that contain speech, compute a voice embedding for each span, then group spans with
similar embeddings into one speaker.

First, a "conversation": three people (george, lucas, theo) take 12 turns, each turn 2–4 digits drawn from the test
recordings, with pauses of 0.15–0.45 s within a turn and 0.5–0.9 s between turns. Three turns get an extra 40 ms
"click" (like a breath or a bump). Every recording is brought to the same RMS level, as if the three shared one
microphone, plus a very faint background noise (−60 dBFS).

In [ ]:
CAST = ("george", "lucas", "theo")
ORDER = (0, 1, 0, 2, 1, 2, 0, 1, 2, 0, 2, 1)  # speaker of each of the 12 turns
CLICK_TURNS = (2, 6, 9)  # turns with a click between their first two digits
rng = np.random.default_rng(0)
pool = {who: list(rng.permutation(TEST[spk[TEST] == SPEAKERS.index(name)])) for who, name in enumerate(CAST)}
parts, truth, clicks = [np.zeros(int(0.3 * SR))], [], []  # truth: (speaker, start, end, recording)


def now():
    return sum(len(p) for p in parts) / SR


for turn, who in enumerate(ORDER):
    n = int(rng.integers(2, 5))
    for c in range(n):
        clip = pool[who].pop()
        truth.append((who, now(), now() + len(waves[clip]) / SR, clip))
        parts.append(waves[clip] * 0.05 / np.sqrt(np.mean(waves[clip] ** 2)))
        if c == n - 1:
            break
        gap = int(rng.uniform(0.15, 0.45) * SR)
        if turn in CLICK_TURNS and c == 0:
            parts.append(np.zeros(gap // 2))
            clicks.append(now())
            parts.append(rng.normal(0, 0.04, 320) * np.hanning(320))  # 40 ms
            parts.append(np.zeros(gap - gap // 2))
        else:
            parts.append(np.zeros(gap))
    parts.append(np.zeros(int(rng.uniform(0.5, 0.9) * SR)))
rec = np.concatenate(parts)
rec = stage1(rec + rng.normal(0, 1e-3, len(rec)), always_peak=False)
print(f"recording {num(len(rec) / SR, 1)} s long: {len(ORDER)} turns, {len(truth)} digits, {len(clicks)} clicks")
play(rec, "the three-person conversation")

**Energy-based speech detection.** Cut the recording into 10 ms frames and compute each frame's energy in dB; a frame
within 35 dB of the loudest frame "has speech". Adjacent speech frames form a span, and two spans less than 80 ms
apart are joined (a break inside a word). This is the simplest voice activity detector (VAD); the report denoises
before stage 2 precisely to make this step more accurate.

In [ ]:
def vad(x, frame=80, floor_db=35, bridge=0.08):
    n = len(x) // frame
    e = 10 * np.log10(np.mean(x[: n * frame].reshape(n, frame) ** 2, 1) + 1e-12)
    on = np.concatenate([[False], e > e.max() - floor_db, [False]])
    starts, ends = np.where(on[1:] & ~on[:-1])[0], np.where(~on[1:] & on[:-1])[0]
    out = []
    for a, b in zip(starts * frame / SR, ends * frame / SR, strict=True):
        if out and a - out[-1][1] < bridge:
            out[-1][1] = b
        else:
            out.append([a, b])
    return [tuple(p) for p in out]


spans = vad(rec)
lens = np.array([b - a for a, b in spans])
print(
    f"{len(spans)} speech spans; {int((lens < 0.1).sum())} shorter than 0.1 s (shortest {num(lens.min() * 1000, 0)} ms)"
)

**Clustering voice embeddings.** Topic 05's voice judge gives each span a 128-dimensional vector. We do *not* use its
6-way output: a real diarizer does not know anyone's name in advance, it only asks "do these two spans have the same
voice?". Bottom-up clustering: each span starts as its own cluster, then we repeatedly merge the two clusters with the
highest average cosine, until that cosine drops below a threshold θ = 0.9. Clusters are named in order of
appearance.

In [ ]:
def embed(xs):
    with torch.no_grad():
        return F.normalize(voice.embed(judge_input(xs)), dim=1).numpy()


def cluster(E, theta):
    """Bottom-up clustering, average linkage on cosine; returns labels in order of appearance."""
    S = E @ E.T
    groups = [[i] for i in range(len(E))]
    while len(groups) > 1:
        sims = {(a, b): S[np.ix_(groups[a], groups[b])].mean() for a in range(len(groups)) for b in range(a)}
        (a, b), best = max(sims.items(), key=lambda kv: kv[1])
        if best < theta:
            break
        groups[b] += groups.pop(a)
    lab = np.empty(len(E), int)
    for k, g in enumerate(sorted(groups, key=min)):
        lab[g] = k
    return lab


def who_said(a, b):
    """The true speaker with the most time inside [a, b] (−1 if nobody: a click)."""
    ov = [(min(e, b) - max(s, a), who) for who, s, e, _ in truth]
    best = max(ov)
    return best[1] if best[0] > 0 else -1


span_x = [rec[int(a * SR) : int(b * SR)] for a, b in spans]
E_spans = embed(span_x)
THETA = 0.9
lab = cluster(E_spans, THETA)
D = [(f"SPEAKER_{k:02d}", a, b) for k, (a, b) in zip(lab, spans, strict=True)]
true_of = np.array([who_said(a, b) for a, b in spans])
ok = lens >= 0.1
for k in sorted(set(lab)):
    m = lab == k
    names = sorted({CAST[w] if w >= 0 else "click" for w in true_of[m & ok]}) or ["only pieces < 0.1 s"]
    print(f"SPEAKER_{k:02d}: {int(m.sum()):2d} spans ({int((m & ~ok).sum())} under 0.1 s); true speakers: "
          f"{', '.join(names)}")  # fmt: skip
with torch.no_grad():
    named = voice(judge_input(span_x)).argmax(-1).numpy()
hit = np.mean([SPEAKERS[n] == CAST[w] for n, w in zip(named[ok], true_of[ok], strict=True)])
print(f"\nfor comparison: naming each span with the 6-speaker output is right for only {pct(hit)} of spans ≥ 0.1 s")

Three people, three clusters, and no cluster mixes two people. A fourth cluster, SPEAKER_02, holds only the 7 pieces
shorter than 0.1 s: the three clicks and a few fragments at the start or end of a word. Their embeddings look like
nobody's, so they stand alone; stage 2 will drop them. Note the last line: with the same network, using the "which of
the 6 people is this" output directly is wrong for a sizeable share of spans (it often mistakes lucas for yweweler),
while the question "do these two spans have the same voice" is much easier. That is also why the report's labels
only mean something within one recording.

**Merging** (pp. 11–12): drop every piece shorter than τ_min = 0.1 s, then walk through time and merge all adjacent
segments with the same label, *regardless of the silence between them* (a silence between two segments of the same
speaker is a natural pause within a turn). Finally, cap at one hour from the start of the first segment: the last
segment is cut at that mark, and dropped if it is then shorter than τ_min.

### 🤔 Predict first
Does the order of "drop short pieces" and "merge" matter? If we merge first and drop the short pieces afterwards,
how many segments does our 12-turn conversation end up with?

In [ ]:
def consolidate(D, tau_min=0.1, cap=3600.0, drop_first=True):
    """Stage 2: drop pieces < τ_min, merge adjacent same-label segments, cap at cap seconds from the first start."""
    seq = [d for d in D if d[2] - d[1] >= tau_min] if drop_first else D
    A = []
    for k, a, b in seq:
        if A and A[-1][0] == k:
            A[-1][2] = b
        else:
            A.append([k, a, b])
    if not drop_first:
        A = [x for x in A if x[2] - x[1] >= tau_min]
    t_lim = A[0][1] + cap
    out = []
    for k, a, b in A:
        if a >= t_lim:
            break
        b = min(b, t_lim)
        if b - a >= tau_min:
            out.append((k, a, b))
    return out


SEGS = consolidate(D)
alt = consolidate(D, drop_first=False)
print(f"{len(spans)} spans → drop short pieces, then merge: {len(SEGS)} segments · merge first, then drop: "
      f"{len(alt)} segments (the conversation has {len(ORDER)} turns)")  # fmt: skip
pure = [len({w for w in true_of[(np.array([a for a, _ in spans]) >= s - 1e-9) & (lens >= 0.1) &
            (np.array([b for _, b in spans]) <= e + 1e-9)]}) == 1 for _, s, e in SEGS]  # fmt: skip
print(f"segments with a single true speaker: {sum(pure)}/{len(SEGS)}")
assert len(SEGS) == len(ORDER) and all(pure)

Dropping first, then merging, gives exactly 12 segments, each one a whole turn of one speaker. Merging first gives
more segments: a click or a fragment with a different label sits *between* two digits of the same speaker, so it
blocks the merge; dropping it afterwards is too late, the turn has already been cut in two. The report's order (drop
first, merge second) is what keeps stray fragments from chopping long turns apart.

The figure below draws all three layers on one time axis: the waveform, the speech spans colored by cluster (pieces
< 0.1 s hatched), and the segments after merging, each with its label. The bottom row is the true speaker.

In [ ]:
COL = {0: BLUE, 1: ORANGE, 2: GREEN, -1: GREY}
lab_name = {}  # each cluster's color: that of the true speaker most of it belongs to (grey for fragments only)
for k in set(lab):
    w = true_of[(lab == k) & ok]
    w = w[w >= 0]
    lab_name[k] = COL[int(np.bincount(w).argmax())] if len(w) else GREY
fig, axes = plt.subplots(4, 1, figsize=(10, 4.6), sharex=True, gridspec_kw={"height_ratios": [1.4, 1, 1, 1]})
tt = np.arange(len(rec)) / SR
axes[0].plot(tt, rec, color=GREY, lw=0.4)
axes[0].set_ylabel("waveform", fontsize=8)
for (a, b), k, short in zip(spans, lab, lens < 0.1, strict=True):
    axes[1].axvspan(a, b, 0.2, 0.8, color=lab_name[k], alpha=0.5 if short else 0.9, hatch="////" if short else None)
for k, a, b in SEGS:
    axes[2].axvspan(a, b, 0.15, 0.85, color=lab_name[int(k[-2:])], alpha=0.85)
    axes[2].text((a + b) / 2, 0.5, k[-2:], ha="center", va="center", fontsize=7, color="white")
for who, a, b, _ in truth:
    axes[3].axvspan(a, b, 0.2, 0.8, color=COL[who], alpha=0.9)
for t0 in clicks:
    axes[3].axvline(t0, color="k", lw=1)
for ax, name in zip(axes[1:], ("speech spans", "after merging", "true speaker"), strict=True):
    ax.set_yticks([])
    ax.set_ylabel(name, fontsize=8, rotation=0, ha="right", va="center")
for who, name in enumerate(CAST):
    axes[3].plot([], [], "s", color=COL[who], label=name)
axes[3].plot([], [], "|", color="k", label="click")
axes[3].legend(loc="upper center", bbox_to_anchor=(0.5, -0.6), ncol=4, fontsize=8, frameon=False)
axes[3].set_xlabel("time (s)")
fmt_axes(fig)
plt.tight_layout()
plt.show()

The one-hour cap never reaches our 35-second recording (section 12 tries a shorter cap). On real data, it keeps a
three-hour podcast episode from becoming one giant "unit".

## 6 · Stage 3: transcribe, then check the transcript

The report transcribes each segment with MOSS-Transcribe-Diarize, the team's ASR that transcribes and marks speakers
at once: each sentence starts with a speaker tag *local to that segment* ([S1], [S2]…), and there may be sound tags
such as [music] or [laugh] (p. 12). We simulate it: the digit judge transcribes each speech span of at least 0.1 s;
shorter spans become a [noise] tag; and the speaker tags come from the true speakers, i.e. an ASR that hears
speakers perfectly.

In [ ]:
def transcribe(seg_a, seg_b):
    """A MOSS-Transcribe-Diarize-style transcript of segment [seg_a, seg_b]."""
    inside = [i for i, (a, b) in enumerate(spans) if a >= seg_a - 1e-9 and b <= seg_b + 1e-9]
    with torch.no_grad():
        words = asr(judge_input([span_x[i] for i in inside])).argmax(-1).numpy()
    tags, out, cur = {}, [], None
    for i, w in zip(inside, words, strict=True):
        if lens[i] < 0.1:
            out.append("[noise]")
            continue
        tag = tags.setdefault(true_of[i], f"[S{len(tags) + 1}]")
        if tag != cur:
            out.append(tag)
            cur = tag
        out.append(WORDS[w])
    return " ".join(out)


TX = [transcribe(a, b) for _, a, b in SEGS]
truth_words = [" ".join(WORDS[digit[c]] for _, s, e, c in truth if min(e, b) > max(s, a)) for _, a, b in SEGS]
n_ok = sum(re.sub(r"\[[^\]]*\]\s*", "", t) == w for t, w in zip(TX, truth_words, strict=True))
for (k, a, b), tx in list(zip(SEGS, TX, strict=True))[:4]:
    print(f"{k} {num(a, 2):>6s}–{num(b, 2):>6s} s: {tx}")
print(f"… every digit transcribed correctly in {n_ok}/{len(SEGS)} segments")

Then the checks. **Three cheap rules** run first, to save LLM calls (p. 12): an empty transcript; a phrase repeated
back to back more than 6 times (a sign of a stuck ASR); after removing every bracketed tag, the remaining text is
under 20% of the transcript's length (a segment that is almost all music or noise). Then the report uses **an LLM**
to diagnose two fatal errors (whole sentences repeated, truncated sentences such as a trailing hyphen) and to repair
in three steps (remove sound tags, remove speaker tags with no words left, fix the format). We have no LLM, so we do
only the parts that rules can do: hyphen truncation, and the first two repairs. Last comes the **single-speaker
check**: the repaired transcript may contain only the [S1] tag. This is the same logic as card 2 in the page's
Practice tab.

In [ ]:
def longest_repeat(words, max_len=4):
    """The most back-to-back repetitions of a 1–4 word phrase."""
    best = 1 if words else 0
    for k in range(1, max_len + 1):
        for i in range(len(words) - k + 1):
            n = 1
            while words[i + n * k : i + (n + 1) * k] == words[i : i + k]:
                n += 1
            best = max(best, n)
    return best


def stage3(raw):
    """Returns (keep?, reason, repaired transcript), in the report's order."""
    no_tags = re.sub(r"\[[^\]]*\]", " ", raw)
    words = [re.sub(r"[^\w]", "", w) for w in no_tags.lower().split()]
    words = [w for w in words if w]
    total, speech = len(re.sub(r"\s", "", raw)), len(re.sub(r"\s", "", no_tags))
    if not raw.strip():
        return False, "rule 1: empty", ""
    if longest_repeat(words) > 6:
        return False, "rule 2: repeated over 6 times", ""
    if speech / total < 0.2:
        return False, "rule 3: under 20% words", ""
    clean = re.sub(r"\[(?!S\d+\])[^\]]*\]", "", raw)  # repair 1: remove sound tags
    clean = re.sub(r"\[S\d+\]\s*(?=\[S\d+\]|$)", "", clean)  # repair 2: remove speaker tags with no words left
    clean = re.sub(r"\s+", " ", clean).strip()
    if re.search(r"[-–]\s*$", clean):
        return False, "LLM: truncated sentence", clean
    tags = set(re.findall(r"\[S\d+\]", clean))
    if tags != {"[S1]"}:
        return False, f"one speaker: has {', '.join(sorted(tags))}", clean
    return True, "keep", clean


EXAMPLES = {
    "clean": "[S1] Today we continue with chapter three. [music]",
    "empty": "  ",
    "repeated": "[S1] yes yes yes yes yes yes yes",
    "all music": "[S1] [music] [laugh] [music] mm",
    "truncated": "[S1] This sentence is cut off in the mid-",
    "two people": "[S1] Hello there. [S2] Hi.",
}
for name, raw_tx in EXAMPLES.items():
    keep, why, clean = stage3(raw_tx)
    print(f"{name:10s} → {why:30s} {clean!r}" if clean else f"{name:10s} → {why}")
assert [stage3(x)[0] for x in EXAMPLES.values()] == [True, False, False, False, False, False]
res3 = [stage3(x) for x in TX]
print(f"\nthe conversation's {len(SEGS)} segments: {sum(r[0] for r in res3)} kept; a repaired example: {res3[2][2]!r}")

All our segments pass: stage 2 did its job, so the [S1] check has nothing to catch. It is a **safety net** for stage
2's mistakes. To see it catch something, we break the diarizer: lower the clustering threshold to θ = 0.7, so that two
people with similar voices end up in one cluster.

In [ ]:
lab_bad = cluster(E_spans, 0.7)
SEGS_BAD = consolidate([(f"SPEAKER_{k:02d}", a, b) for k, (a, b) in zip(lab_bad, spans, strict=True)])
res_bad = [stage3(transcribe(a, b)) for _, a, b in SEGS_BAD]
print(f"θ = 0.7: {len(set(lab_bad[ok]))} clusters for 3 people, {len(SEGS_BAD)} segments; the single-speaker check "
      f"drops {sum(not r[0] for r in res_bad)}")  # fmt: skip
for (k, a, b), r in zip(SEGS_BAD, res_bad, strict=True):
    if not r[0]:
        print(f"  {k} {num(a, 1)}–{num(b, 1)} s: {r[1]} · {r[2][:70]}")
assert any(not r[0] for r in res_bad)

With a loose threshold, stage 2 merges the turns of two people into one long segment; the ASR hears two voices and
writes an extra [S2], so stage 3 drops the segment. On real data, the diarizer and the ASR are two different models,
so one's mistakes have a chance to be caught by the other. (Here the ASR hears speakers "perfectly", so our safety
net catches every mistake; a real ASR does not.)

## 7 · Stage 4: character rate

Stage 4 (p. 13) scores audio quality on the *undenoised* audio (DNSMOS > 2.8, AudioBox PQ > 6.5), requires the
language Whisper detects from the audio to match the label an LLM infers from the text, and checks the **character
rate** r = |x′| / d: the number of characters in the repaired transcript divided by the segment's length in seconds.
A segment that is too slow is often silence or background sound; one that is too fast is often an ASR that made up
words. The valid range [r_min, r_max] for each language comes from the statistics of a reference corpus; the report
does not publish the ranges.

We can only do the last part. Our reference corpus is FSDD's training recordings (numbered 10 and up), and the valid
range runs from their 1st to their 99th percentile, a choice of this notebook alone. We count characters without
spaces.

In [ ]:
def char_rate(text, seconds):
    return len(re.sub(r"\[[^\]]*\]|\s", "", text)) / seconds


r_clip = np.array([char_rate(WORDS[d], s) for d, s in zip(digit, dur, strict=True)])
r_min, r_max = np.percentile(r_clip[idx >= 10], [1, 99])
bad = TEST[(r_clip[TEST] < r_min) | (r_clip[TEST] > r_max)]
print(f"valid range (1st–99th percentile of the training recordings): {num(r_min, 1)}–{num(r_max, 1)} characters/s")
print(f"dropped among the {len(TEST)} test recordings: {len(bad)}")
for c in bad[np.argsort(r_clip[bad])][[0, -1]]:
    print(f"  {SPEAKERS[spk[c]]} saying {WORDS[digit[c]]!r} in {num(dur[c], 2)} s: r = {num(r_clip[c], 1)}")
r_seg = np.array([char_rate(c, b - a) for (_, a, b), (_, _, c) in zip(SEGS, res3, strict=True)])
print(f"the conversation's 12 segments: r = {num(r_seg.min(), 1)}–{num(r_seg.max(), 1)} characters/s")

The figure below is the distribution of r over the 3,000 recordings (bars), the two lines are the valid range, and
the dots are the conversation's 12 segments. The slowest dropped recording usually has a long silence at its start
or end; listen to it in the next cell.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(r_clip, bins=60, color=BLUE, alpha=0.7, label="3,000 FSDD recordings")
for v in (r_min, r_max):
    ax.axvline(v, color=ORANGE, ls="--", lw=1.5)
ax.plot(r_seg, np.full(len(r_seg), 5), "D", color=GREEN, ms=6, label="12 conversation segments")
ax.text(r_min, ax.get_ylim()[1] * 0.92, " r_min", color=ORANGE, fontsize=8)
ax.text(r_max, ax.get_ylim()[1] * 0.92, " r_max", color=ORANGE, fontsize=8)
ax.set(xlabel="character rate r (characters per second)", ylabel="recordings")
ax.legend(fontsize=8, frameon=False)
fmt_axes(fig)
plt.tight_layout()
plt.show()
slow = bad[np.argmin(r_clip[bad])]
play(waves[slow], f"dropped as too slow: {SPEAKERS[spk[slow]]} saying {WORDS[digit[slow]]!r}, {num(dur[slow], 2)} s")

The conversation segments are much slower than the single recordings, because they include the pauses between
digits, but they still fall inside the range. That shows why the report takes its ranges from a reference corpus
*of the same kind* as the data it filters: ranges from one-word recordings applied to continuous speech, or the other
way round, would wrongly drop a great deal.

## 8 · Synthetic data: choosing voice prompts

Web data gives (audio, transcript) pairs, but no **voice prompt**: nothing teaches the model to listen to a short
clip and go on speaking in that voice. The report builds (prompt, target) pairs from the filtered data itself
(pp. 13–14): group each speaker's segments within one recording, s₁ … sₙ. For each target segment sᵢ, cut 5 random
crops from *every* other segment sⱼ (start and end drawn independently, at most 30 s long), which gives 5(n − 1)
candidates; score the cosine between each candidate's voice embedding and sᵢ's (with the WavLM-Large of
Seed-TTS-eval), and take the highest-scoring crop as the prompt. We use the voice judge's embeddings instead of
WavLM, and add one condition of this notebook's own: a crop lasts at least 0.25 s, so the judge has something to
listen to.

In [ ]:
def crops(a, b, n, rng, max_s=30.0, min_s=0.25):
    out = []
    while len(out) < n:
        s, e = np.sort(rng.uniform(a, b, 2))
        if e - s >= min_s:
            out.append((s, min(e, s + max_s)))
    return out


by_spk = {}
for k, a, b in SEGS:
    by_spk.setdefault(k, []).append((a, b))
rng = np.random.default_rng(1)
PAIRS = []
for k, segs in by_spk.items():
    E_t = embed([rec[int(a * SR) : int(b * SR)] for a, b in segs])
    for i in range(len(segs)):
        cand = [(j, c) for j, (a, b) in enumerate(segs) if j != i for c in crops(a, b, 5, rng)]
        assert len(cand) == 5 * (len(segs) - 1)
        sim = embed([rec[int(s * SR) : int(e * SR)] for _, (s, e) in cand]) @ E_t[i]
        PAIRS.append({"spk": k, "target": segs[i], "cand": cand, "sim": sim, "best": int(sim.argmax())})
print("segments per speaker:", ", ".join(f"{k}: {len(v)}" for k, v in by_spk.items()))
print(f"each target has {len(PAIRS[0]['cand'])} candidates = 5 × (4 − 1), each crop {num(0.25, 2)}–30 s long; "
      f"{len(PAIRS)} pairs in all")  # fmt: skip
assert all(len(v) == 4 for v in by_spk.values()) and len(PAIRS) == 12

### 🤔 Predict first
Among the 15 candidates, is the winning crop always the longest one? What does the worst crop look like?

In [ ]:
won_longest = 0
for p in PAIRS:
    L_c = np.array([e - s for _, (s, e) in p["cand"]])
    won_longest += int(p["best"] == L_c.argmax())
    p["len"] = L_c
best_sims = np.array([p["sim"][p["best"]] for p in PAIRS])
worst_sims = np.array([p["sim"].min() for p in PAIRS])
print(f"cosine of the winning crop: {num(best_sims.min(), 3)}–{num(best_sims.max(), 3)}; "
      f"of the worst crop: {num(worst_sims.min(), 3)}–{num(worst_sims.max(), 3)}")  # fmt: skip
print(f"the winner is the longest crop for {won_longest}/{len(PAIRS)} targets")
other = []
for p in PAIRS:
    a, b = p["target"]
    e_t = embed([rec[int(a * SR) : int(b * SR)]])[0]
    rival = [x for k, segs in by_spk.items() if k != p["spk"] for x in segs]
    other.append(max(embed([rec[int(s * SR) : int(e * SR)] for s, e in rival]) @ e_t))
print(
    f"for comparison: highest cosine with a whole segment of *another* speaker: {num(min(other), 3)}–{num(max(other), 3)}"
)

The winning crop is usually long, but not always the longest; and the worst crop has a much lower cosine, sometimes
lower than a segment of another speaker, because a random crop can fall almost entirely into the pause between two
digits. That is why the report scores *each crop* rather than whole segments: what the model receives at inference
is a short clip, so we must pick the short clip that carries the voice best.

The figure below is one target segment: each dot is a candidate, the horizontal axis is the crop's length and the
vertical axis its cosine; the marker shape tells which segment the crop was cut from, and the large circle is the
winner. Listen to the prompt and the target right after.

In [ ]:
P0 = PAIRS[0]
fig, ax = plt.subplots(figsize=(7, 3))
marks = {}
for (j, _), L_c, s in zip(P0["cand"], P0["len"], P0["sim"], strict=True):
    m = marks.setdefault(j, "osD^"[len(marks)])
    ax.plot(L_c, s, m, color=BLUE, ms=7, alpha=0.8)
ax.plot(P0["len"][P0["best"]], P0["sim"][P0["best"]], "o", ms=16, mfc="none", mec=GREEN, mew=2)
for j, m in marks.items():
    ax.plot([], [], m, color=BLUE, label=f"crop from segment {j + 1} of {P0['spk']}")
ax.plot([], [], "o", ms=12, mfc="none", mec=GREEN, mew=2, label="winner")
ax.set(xlabel="crop length (s)", ylabel="cosine with the target")
ax.legend(fontsize=8, frameon=False, loc="lower right")
fmt_axes(fig)
plt.tight_layout()
plt.show()
j, (s, e) = P0["cand"][P0["best"]]
play(rec[int(s * SR) : int(e * SR)], f"voice prompt: {num(e - s, 2)} s cut from segment {j + 1}")
a, b = P0["target"]
play(rec[int(a * SR) : int(b * SR)], f"target: {num(b - a, 2)} s")

**Two copies of each example.** So that the model also learns duration control, each training example is written
twice (p. 14): once with a `tokens` field equal to the exact number of audio tokens of the target, once with that
field set to `None`, leaving the length to the model. The tokenizer gives 12.5 frames per second (p. 20), so the
token count is 12.5 · d.

In [ ]:
FPS = 24000 / (240 * 2 * 2 * 2)  # 12.5 frames per second (section 9)
a, b = P0["target"]
text0 = res3[[x[1] for x in SEGS].index(a)][2].removeprefix("[S1] ")
for tokens in (round(FPS * (b - a)), None):
    print({"prompt": f"<{num(e - s, 2)} s voice prompt>", "text": text0, "tokens": tokens})
assert round(FPS * 10) == 125

The report also adds three small kinds of data that change only the text and keep the audio: text with noisy
punctuation and whitespace (real users type "Hello??!!   are you there"), text written phonetically (pinyin with
tone marks for Chinese, IPA between slashes for English), and single-word sentences. They are string
transformations, so we do not repeat them here.

## 9 · The tokenizer: counting

MOSS-Audio-Tokenizer (pp. 4–7) is the CAT tokenizer that Topic 05 rebuilds at a small scale. At full size: 24,000
samples per second, patches of 240 then three times 2, so each frame is 1,920 samples; RVQ with 32 layers of 1,024
codes each.

In [ ]:
SR_TOK, PATCHES, NQ, CODES = 24000, (240, 2, 2, 2), 32, 1024
hop = math.prod(PATCHES)
ladder = [SR_TOK]
for p in PATCHES:
    ladder.append(ladder[-1] / p)
bits = math.log2(CODES)
print(" → ".join(num(v, 1 if v % 1 else 0) for v in ladder), "positions per second")
print(
    f"{num(hop, 0)} samples per frame; {num(FPS * bits, 0)} bits/s per layer; {num(FPS * bits * NQ, 0)} bits/s for all {NQ}"
)
print(f"tokenizer stage 1: 5 hours of audio per batch of 1,536 examples = {num(5 * 3600 / 1536, 2)} s per example")
assert hop == 1920 and ladder[-1] == FPS == 12.5 and FPS * bits == 125 and FPS * bits * NQ == 4000

The tokenizer is trained separately, before the TTS model, and on broader audio (music and sound effects too):
520,000 steps without a GAN, then 500,000 steps with one (pp. 6–7). How it learns, and why *quantizer dropout* makes
it usable at any number of layers, is the subject of Topic 05.

## 10 · 33 token channels, and the delay pattern

At every step, both of the report's models predict **33 channels** (pp. 7–8): channel 0 is text (on text-only
steps) or a padding symbol (on audio steps), plus 32 audio channels, one per RVQ layer. The problem: the 32 tokens of
one frame are not independent, layer 2 corrects layer 1's error. Predicting all 32 at once from a single state, layer
2 cannot know what layer 1 chose.

The **delay pattern** (MusicGen; pp. 8–9) shifts layer j by j − 1 steps: ã_{j,t} = a_{j,t−(j−1)}. At step t the model
predicts layer 1 of frame t, layer 2 of frame t − 1, and so on: by the time layer j of a frame is due, the earlier
layers of that frame are already in the input. Each step's input is the *sum* of the 32 layers' embeddings, and a
mask m hides the empty cells the shift creates.

We use real tokens: the digit "three" in 32 layers, as generated by Topic 05's small CAT-TTS (stored in the data
file), plus five text tokens t, h, r, e, e (one token per character here, to keep it easy to see).

In [ ]:
SH = W["show"]["0"]
assert SH["k"][0] == NQ and SH["target"][0] == 3
A = torch.tensor(SH["codes"][0][0])  # (T frames, 32 layers)
T = A.shape[0]
PAD = -1


def delay(A):
    """(T, Nq) → (T + Nq − 1, Nq) and a mask: layer j (counting from 0) shifted by j steps."""
    T_, nq = A.shape
    D_ = torch.full((T_ + nq - 1, nq), PAD)
    m = torch.zeros(T_ + nq - 1, nq, dtype=torch.bool)
    for j in range(nq):
        D_[j : j + T_, j] = A[:, j]
        m[j : j + T_, j] = True
    return D_, m


def undelay(D_, T_):
    return torch.stack([D_[j : j + T_, j] for j in range(D_.shape[1])], 1)


Dl, mask = delay(A)
assert torch.equal(undelay(Dl, T), A), "undoing the shift must give back the exact token matrix"
step_of = torch.stack([torch.arange(T) + j for j in range(NQ)], 1)  # the step that produces layer j of frame f
assert (step_of[:, 1:] > step_of[:, :-1]).all(), "within a frame, a later layer is always produced after an earlier one"
print(
    f"{T} frames × {NQ} layers → {Dl.shape[0]} steps × {NQ} audio channels; masked cells: {int((~mask).sum())} = {NQ} × {NQ - 1}"
)
print("frame 0: layers 1, 2, 3, …, 32 are produced at steps", step_of[0, :3].tolist(), "…", step_of[0, -1].item())
assert int((~mask).sum()) == NQ * (NQ - 1)

The figure below is the 33-channel sequence the Transformer reads: 5 text steps at the start (channel 0 is text, the
audio channels are masked), then T + 31 audio steps (channel 0 is padding). Each color is a code; hatched cells are
masked and left out of the loss. The layer 1 row starts right away, the layer 32 row 31 steps later, so the real
cells form a diagonal band.

In [ ]:
TEXT = "three"
n_text = len(TEXT)
codes = np.full((NQ + 1, n_text + Dl.shape[0]), np.nan)  # only real audio cells get a color
codes[1:, n_text:] = np.where(mask.T.numpy(), Dl.T.numpy() % 97 / 97, np.nan)
hidden = np.zeros(codes.shape, bool)
hidden[1:, :n_text] = True  # text steps: the audio channels are masked
hidden[1:, n_text:] = ~mask.T.numpy()
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(codes, aspect="auto", cmap="viridis", interpolation="nearest")
ys, xs = np.nonzero(hidden)
ax.add_collection(PatchCollection([Rectangle((x - 0.5, y - 0.5), 1, 1) for y, x in zip(ys, xs, strict=True)],
                                  facecolor="none", edgecolor=GREY, hatch="////", linewidth=0))  # fmt: skip
for x in range(codes.shape[1]):
    is_text = x < n_text
    ax.add_patch(Rectangle((x - 0.5, -0.5), 1, 1, facecolor=GREEN if is_text else "#d9dde3", edgecolor="white"))
    if is_text:
        ax.text(x, 0, TEXT[x], ha="center", va="center", fontsize=7, color="white")
ax.text(n_text + 0.5, 0, "padding (still a target to predict)", va="center", fontsize=7)
ax.set_yticks([0, 1, 8, 16, 32], ["channel 0 · text/pad", "layer 1", "layer 8", "layer 16", "layer 32"], fontsize=8)
ax.set_xlabel("Transformer step")
ax.axvline(n_text - 0.5, color="k", lw=1)
fmt_axes(fig)
plt.tight_layout()
plt.show()

### 🤔 Predict first
A 10-second sentence is 125 frames. With the delay pattern, how many steps must the large Transformer run to produce
all the audio? With the Local Transformer (a small model that runs inside each frame, pp. 9–10), how many steps does
the large Transformer run, and how many the small one? After how many steps is the first frame complete with 32
tokens?

In [ ]:
def steps_delay(T_, nq=NQ):
    return T_ + nq - 1


def steps_local(T_, nq=NQ):
    return T_, T_ * (nq + 1)  # large steps; small steps (33 channels per frame)


for frames in (T, 100, 125, 750):
    big, small = steps_local(frames)
    print(f"{frames:4d} frames ({num(frames / FPS, 1):>4s} s): delay {steps_delay(frames):4d} large steps, "
          f"masked cells {pct(NQ * (NQ - 1) / (steps_delay(frames) * NQ))} · local {big} large + "
          f"{num(small, 0)} small steps")  # fmt: skip
assert steps_delay(100) == 131 and steps_delay(125) == 156 and steps_delay(1) == 32

The delay pattern adds only 31 steps, however long the sentence: a 10-second sentence takes 156 steps instead of
125, and the share of masked cells shrinks as sentences get longer. Each step needs one pass of the large Transformer
followed by 33 projections, so it is simple and scales easily; the report uses it for its main 8-billion-parameter
model. The price: the first frame is complete only after 32 steps. The Local Transformer shifts nothing: the large
Transformer runs exactly T steps, then a small model produces the 33 channels of the next frame one by one, each
channel seeing the channels before it *within the same frame*. That is many more small steps, but each small step is
cheap; and in the report, the 1.7-billion-parameter local model keeps the voice better than the 8-billion delay model
on Seed-TTS-eval (Table 3, p. 19).

**Weighted loss.** Both models use the same cross-entropy weighted per channel (p. 8): λ = (1, 3, 3, 3, 2, 2, 2, 1,
…, 1). The text/padding channel and the last 26 layers get weight 1; the first three layers 3, the next three 2. The
cell below computes the loss over the 32 audio channels of an "untrained" model (every code equally likely) on the
delay sequence above, with and without the mask, and the share of the weight that the first 6 layers carry.

In [ ]:
LAM = torch.tensor([1, 3, 3, 3, 2, 2, 2] + [1] * 26, dtype=torch.float32)
assert len(LAM) == NQ + 1 and LAM.sum() == 42
logits = torch.zeros(Dl.shape[0], NQ, CODES)  # every code equally likely
ce = F.cross_entropy(logits.reshape(-1, CODES), Dl.clamp_min(0).reshape(-1), reduction="none").view(Dl.shape)
per_layer = (ce * mask).sum(0)  # loss per audio layer, real cells only
loss = (LAM[1:] * per_layer).sum()
print(f"per cell: −log(1/1,024) = {num(math.log(CODES), 3)}; each layer has {T} real cells out of {Dl.shape[0]}")
print(f"masked loss: {num(loss, 1)} = {num(math.log(CODES), 3)} × {T} × {num(LAM[1:].sum(), 0)}")
print(f"without the mask, the empty cells count too: {num((LAM[1:] * ce.sum(0)).sum(), 1)}")
print(f"the first 6 layers are {pct(6 / 33)} of the channels but carry {pct(LAM[1:7].sum() / LAM.sum())} of the weight")
assert abs(loss - math.log(CODES) * T * 41) < 1e-2

Without the mask, the model is penalized for failing to predict cells that *hold no token at all*. With the weights
λ, the first layers (which carry the coarse, most important part of the sound: Topic 05 finds 1 layer is already
enough to hear the digit) get 2–3 times the attention.

## 11 · Four phases and the learning-rate schedule

Pretraining has four phases (Table 1, pp. 15–16), in the **warmup–stable–decay** style (WSD, as in MiniCPM):

| Phase | Longest sequence | Learning rate | Data |
|---|---|---|---|
| 1 | 32k | warm up to 2·10⁻⁴, then hold | filtered data only, $\mathcal{D}_{\text{basic}}$ |
| 2 | 32k | hold 2·10⁻⁴ | all sets; voice prompts $\mathcal{D}_{\text{clone}}$ sampled far more often |
| 3 | 32k | linear decay 2·10⁻⁴ → 2·10⁻⁶ | all sets, normal proportions |
| 4 | 64k | hold 2·10⁻⁶ | more long data, sequences extended to 64k |

The report does not publish the phase lengths. We draw the schedule with four equal phases and a warmup of 15% of
phase 1, like card 5 in the Practice tab; that is an assumption, not a number from the report.

In [ ]:
LR_PEAK, LR_END = 2e-4, 2e-6
WARM = 0.15  # warmup share of phase 1; the report gives no value


def lr_at(u, peak=LR_PEAK, end=LR_END, lengths=(1, 1, 1, 1), warm=WARM):
    """Learning rate at progress u ∈ [0, 1] of the four-phase schedule."""
    e = np.cumsum([0, *lengths]) / sum(lengths)
    if u < e[1] * warm:
        return peak * u / (e[1] * warm)
    if u < e[2]:
        return peak
    if u < e[3]:
        return peak + (end - peak) * (u - e[2]) / (e[3] - e[2])
    return end


print(
    f"peak {sci(LR_PEAK)}, end {sci(LR_END)}: {num(LR_PEAK / LR_END, 0)} times lower, and the decay is only in phase 3"
)
print(f"the notebook's assumption: four equal phases, warmup over the first {pct(WARM, 0)} of phase 1")
assert round(LR_PEAK / LR_END) == 100 and lr_at(0.5) == LR_PEAK and lr_at(0.9) == LR_END

Why hold the learning rate high for two phases before decaying? With a high learning rate, each update keeps
bouncing around the bottom instead of reaching it, so the loss stalls at a "noisy" level. The decay gathers those
bounces in. We try it on a problem small enough to run in a few seconds: softmax regression that guesses the digit
from 80 log-mel features (the mean and standard deviation of 40 bands) of each FSDD recording, Adam, batch 16, 2,000
steps, 3 seeds. Both schedules share the peak 0.03: held throughout, or four phases as above (peak/100 at the end).

In [ ]:
Xf = np.zeros((len(waves), 10240), np.float32)
for b, x in enumerate(waves):
    x = x[:10240]
    Xf[b, : len(x)] = x * 0.1 / np.sqrt(np.mean(x**2))
with torch.no_grad():
    M = logmel(torch.tensor(Xf))
    act = (M.max(1).values > -3).float()[:, None]  # speech frames only
    mu = (M * act).sum(-1) / act.sum(-1)
    sd = (((M - mu[..., None]) ** 2) * act).sum(-1).div(act.sum(-1)).sqrt()
    feats = torch.cat([mu, sd], 1)
TRN = np.where(idx >= 5)[0]
feats = (feats - feats[TRN].mean(0)) / feats[TRN].std(0)
y_d = torch.tensor(digit)
STEPS, PEAK, BS = 2000, 0.03, 16


def run(schedule, seed, steps=STEPS, peak=PEAK, bs=BS, lengths=(1, 1, 1, 1)):
    g = torch.Generator().manual_seed(seed)
    Wl, bl = torch.zeros(80, 10, requires_grad=True), torch.zeros(10, requires_grad=True)
    opt = torch.optim.Adam([Wl, bl], lr=peak)
    curve = []
    for k in range(steps):
        lr = lr_at(k / steps, peak, peak / 100, lengths) if schedule == "wsd" else peak
        for pg in opt.param_groups:
            pg["lr"] = lr
        i = torch.tensor(TRN)[torch.randint(len(TRN), (bs,), generator=g)]
        loss = F.cross_entropy(feats[i] @ Wl + bl, y_d[i])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (k + 1) % 50 == 0:
            with torch.no_grad():
                curve.append(float(F.cross_entropy(feats[TRN] @ Wl + bl, y_d[TRN])))
    with torch.no_grad():
        acc = float(((feats[TEST] @ Wl + bl).argmax(1) == y_d[TEST]).float().mean())
    return np.array(curve), acc


RUNS = {s: [run(s, seed) for seed in range(3)] for s in ("const", "wsd")}
print(
    f"Adam, batch {BS}, {num(STEPS, 0)} steps, 3 seeds; peak {num(PEAK, 2)} (the four-phase schedule ends at {num(PEAK / 100, 4)})"
)
for s, name in (("const", "held"), ("wsd", "four phases")):
    c = np.array([r[0] for r in RUNS[s]])
    accs = [r[1] for r in RUNS[s]]
    print(f"{name:11s} training loss at 50%: {num(c[:, 19].min(), 3)}–{num(c[:, 19].max(), 3)} · at 75%: "
          f"{num(c[:, 29].min(), 3)}–{num(c[:, 29].max(), 3)} · final: {num(c[:, -1].min(), 3)}–{num(c[:, -1].max(), 3)} · "
          f"test accuracy {pct(min(accs))}–{pct(max(accs))}")  # fmt: skip
cw, cc = np.array([r[0] for r in RUNS["wsd"]]), np.array([r[0] for r in RUNS["const"]])
assert cw[:, -1].max() < cc[:, -1].min(), "four phases must end lower than the held schedule, on all 3 seeds"

### 🤔 Predict first
The figure below plots the learning-rate schedules (top) and the training loss of both (bottom). Before looking: when
the learning rate starts to decay at the start of phase 3 (50% progress), does the four-phase loss go up, stay put or
drop?

In [ ]:
u = np.arange(1, 41) * 50 / STEPS
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True, gridspec_kw={"height_ratios": [1, 1.6]})
uu = np.linspace(0, 1, 401)
a1.plot(uu, [lr_at(v, PEAK, PEAK / 100) for v in uu], color=BLUE, lw=2, label="four phases")
a1.plot(uu, np.full_like(uu, PEAK), color=ORANGE, lw=2, ls="--", label="held")
a1.set(ylabel="learning rate", ylim=(0, 1.23 * PEAK))
a1.legend(fontsize=8, frameon=False)
for s, col, ls in (("wsd", BLUE, "-"), ("const", ORANGE, "--")):
    for c, _ in RUNS[s]:
        a2.plot(u, c, color=col, ls=ls, lw=1.2, alpha=0.85)
a2.set(yscale="log", xlabel="training progress", ylabel="training loss")
for ax in (a1, a2):
    for x in (0.25, 0.5, 0.75):
        ax.axvline(x, color=GREY, lw=0.8, ls=":")
for i in range(4):
    a1.text(0.125 + 0.25 * i, 1.1 * PEAK, f"phase {i + 1}", ha="center", fontsize=8)
fmt_axes(fig)
plt.tight_layout()
plt.show()

The four-phase loss drops during phase 3, as the learning rate decays, then stays put in phase 4; the held schedule
keeps bouncing around a higher level to the end. That is what the report calls phase 3 "consolidating quality": most
of the final gain arrives during the decay. WSD's practical benefit is that we can hold the learning rate high as
long as we like (adding data, such as $\mathcal{D}_{\text{clone}}$ in phase 2) and only then decay, instead of
having to fix the total number of steps in advance as with a cosine schedule. Our problem is convex and small, so it
only illustrates the shape of the effect, not why the report chose to bring in voice prompts early or to extend
sequences to 64k late.

## 12 · Things to try

### The clustering threshold θ
θ too high splits one person into several labels; θ too low lumps several people into one. The cell below sweeps θ
and counts the clusters (spans ≥ 0.1 s only), the segments after merging and the segments the single-speaker check
drops.

In [ ]:
for theta in (0.6, 0.7, 0.8, 0.85, 0.9, 0.94, 0.96, 0.98):
    lb = cluster(E_spans, theta)
    sg = consolidate([(f"SPEAKER_{k:02d}", a, b) for k, (a, b) in zip(lb, spans, strict=True)])
    dropped = sum(not stage3(transcribe(a, b))[0] for _, a, b in sg)
    print(f"θ = {num(theta, 2)}: clusters {len(set(lb[ok])):2d}, segments {len(sg):2d}, dropped by stage 3: {dropped}")

### τ_min and the cap
The report caps at one hour; with our 35-second recording, try a cap of 9 seconds (from the start of the first
segment) to see the last segment cut. Then raise τ_min to 0.5 s: many digits are shorter than half a second, so they
are dropped as fragments.

In [ ]:
sg = consolidate(D, cap=9.0)
k, a, b = sg[-1]
print(f"cap at 9 s: {len(sg)} segments; the last is {k} {num(a, 2)}–{num(b, 2)} s (without the cap it runs to "
      f"{num(SEGS[len(sg) - 1][2], 2)} s)")  # fmt: skip
sg = consolidate(D, tau_min=0.5)
print(f"τ_min = 0.5 s: only {sum(e - s >= 0.5 for _, s, e in D)}/{len(D)} spans are long enough → {len(sg)} segment(s): "
      + ", ".join(f"{k} {num(a, 2)}–{num(b, 2)} s ({num(b - a, 1)} s)" for k, a, b in sg))  # fmt: skip
k, a, b = sg[0]
inside = [CAST[w] for w in sorted({w for w, s, e, _ in truth if s >= a and e <= b})]
print(f"that segment contains speech by {', '.join(inside)}; stage 3: {stage3(transcribe(a, b))[1]}")

With τ_min = 0.5 s, only five of george's digits are long enough to keep. The merge rule ignores silence, so those
five same-label spans become **one** long segment that swallows the turns of lucas and theo lying between them. The
rule "merge all adjacent segments with the same label" is only safe while the other speakers' turns are still in the
sequence to block the merge; a small τ_min (0.1 s) keeps it that way. Here stage 3 still catches the broken
segment, because our ASR hears three voices.

### Fewer crops per segment
The report cuts 5 crops from each other segment. With 1 crop, how much worse is the winner? Change `n_crops` and run
again.

In [ ]:
n_crops = 1
rng = np.random.default_rng(1)
few = []
for k, segs in by_spk.items():
    E_t = embed([rec[int(a * SR) : int(b * SR)] for a, b in segs])
    for i in range(len(segs)):
        cand = [c for j, (a, b) in enumerate(segs) if j != i for c in crops(a, b, n_crops, rng)]
        few.append(max(embed([rec[int(s * SR) : int(e * SR)] for s, e in cand]) @ E_t[i]))
print(f"{n_crops} crop(s) per segment: winning cosine {num(min(few), 3)}–{num(max(few), 3)} "
      f"(5 crops: {num(best_sims.min(), 3)}–{num(best_sims.max(), 3)})")  # fmt: skip

### Phase lengths
The report does not publish the phase lengths. Try a short phase 3 (a fast decay) and a long one, with the same total
number of steps.

In [ ]:
def ratio(lengths):
    return "(" + ", ".join(tick(v) for v in lengths) + ")"


print(f"phase lengths {ratio((1, 1, 1, 1))}: final loss {num(cw[:, -1].min(), 3)}–{num(cw[:, -1].max(), 3)} (above)")
for lengths in ((1, 1, 0.3, 1), (1, 1, 2.5, 0.2)):
    finals = [run("wsd", seed, lengths=lengths)[0][-1] for seed in range(3)]
    print(f"phase lengths {ratio(lengths)}: final loss {num(min(finals), 3)}–{num(max(finals), 3)}")

On this problem, the longer phase 3, the lower the final loss: for the same total number of steps, a decaying step
is worth more than a step held at the peak or at the bottom. The report does not say how it splits its time.

## 13 · Limitations

- **No large model runs here.** Denoising, DNSMOS and AudioBox PQ scoring, the LLM that diagnoses and repairs
  transcripts, Whisper's language detection, WavLM-Large, the 1.6-billion-parameter tokenizer (p. 4) and the 8-billion
  and 1.7-billion models: we only rebuild the computations around them. Topic 05's two judges stand in for the ASR and
  the voice embeddings, and they only know 10 digits and 6 voices.
- **Made-up data.** The conversation is stitched from one-word recordings, at the same level, over a nearly silent
  background. There is no overlapping speech, background music or reverberation, which make real diarization much
  harder.
- **An ASR that hears speakers perfectly.** The transcript's [S1]/[S2] tags come from the true speakers, so section
  6's single-speaker check catches every stage 2 mistake; a real ASR does not.
- **What the report leaves unclear:** the reading of the division by the peak amplitude, the value of ε, the
  [r_min, r_max] ranges, the phase lengths, the warmup length. The notebook picks a value for each and marks it as
  its own choice.
- **Section 11's problem is convex and small.** It shows the shape of the WSD effect, not how large it is for an
  8-billion-parameter model.

## Summary

- **Stage 1:** the gain clip(−20 − L, −3, 3) dB barely narrows FSDD's ~38 dB gap; always dividing by max |x| cancels
  that gain, and the division itself is what crowds the levels together. The report does not say which is meant.
- **Stage 2:** energy-based speech detection, voice-embedding clustering at θ = 0.9, then dropping pieces < 0.1 s
  *before* merging: 12 turns become exactly 12 single-speaker segments. Merging first lets stray fragments cut turns
  into more segments.
- **Stage 3:** three cheap rules, the easy part of the LLM, then the [S1]-only check, a safety net for stage 2's
  merge mistakes.
- **Stage 4:** the character rate drops recordings that are too slow (much silence) or too fast; the valid range must
  come from a corpus of the same kind.
- **Voice prompts:** 5(n − 1) candidates, each crop scored by cosine; the winner is not always the longest.
- **Model:** 33 channels per step; the delay pattern adds exactly 31 steps (156 for 125 frames) and masks 32 × 31
  cells; the Local Transformer runs 125 large steps and 4,125 small ones; the first 6 layers carry 35.7% of the total
  weight λ.
- **Four phases:** hold the learning rate high, then decay it 100 times only in phase 3; on the small problem, the
  loss drops during phase 3 and ends below the held schedule on all 3 seeds.

## Review questions

1. A recording at −14 dBFS peaks at 0.5. After stage 1, what is its RMS level under reading A? Under reading B?
2. Why does the report score DNSMOS and AudioBox PQ on the *undenoised* audio, even though stage 1 denoises it?
3. A speaker has 5 segments in a recording. For one target segment, how many prompt candidates are there? Why not
   simply use a whole other segment as the prompt?
4. With the delay pattern, how many Transformer steps does an 8-second sentence need? How many cells are masked?
5. Removing the mask raises the loss of section 10's "untrained" model. Why is training with an unmasked loss
   harmful, and not merely extra computation?
6. The report brings in $\mathcal{D}_{\text{clone}}$ in phase 2, while the learning rate is still high, then returns
   to normal proportions in phase 3. According to the report, what happens if voice prompts are oversampled for too
   long?

<details><summary>Answer hints</summary>

1. L = −14, so the gain is clamped at −3 dB, the level becomes −17 dBFS and the peak 0.5 · 10^(−3/20) ≈ 0.35.
   Reading A divides by 0.35: the level becomes −17 − 20 log₁₀ 0.35 ≈ −8 dBFS (that is, −14 − 20 log₁₀ 0.5, as if
   there were no gain). Reading B: the peak does not exceed 1, so it stays at −17 dBFS.
2. The denoiser can make quality scores look better: the score would then reflect the denoiser rather than the
   original recording. Stage 1's denoising is only there to make diarization more accurate, not as a training signal.
3. 5 · (5 − 1) = 20. At inference, the model receives an arbitrary short clip; scoring each crop picks the short clip
   that carries the voice best, and the random length and position (up to 30 s) teach the model to cope with prompts
   of different lengths.
4. 8 seconds is 100 frames, so 100 + 31 = 131 steps; the number of masked cells is always 32 · 31 = 992.
5. The model would be taught to predict cells that hold no real token (for example, to predict "padding" there),
   i.e. to learn a regularity of the layout rather than of the audio, and that share of the weight takes attention
   away from the real cells.
6. The model leans toward copying the prompt (*prompt copying*), and its other abilities (ordinary multilingual TTS,
   reading single words, coping with noisy text) get crowded out (p. 15).
</details>

## Further reading

- SII-OpenMOSS (2026), *MOSS-TTS Technical Report*: [arXiv:2603.18090](https://arxiv.org/abs/2603.18090); code and
  models: [OpenMOSS/MOSS-TTS](https://github.com/OpenMOSS/MOSS-TTS).
- Gong et al. (2026), *MOSS-Audio-Tokenizer*: [arXiv:2602.10934](https://arxiv.org/abs/2602.10934).
- Copet et al. (2023), *Simple and Controllable Music Generation* (MusicGen, the delay pattern):
  [arXiv:2306.05284](https://arxiv.org/abs/2306.05284).
- Défossez et al. (2024), *Moshi* (an RQ-Transformer over time and over depth):
  [arXiv:2410.00037](https://arxiv.org/abs/2410.00037).
- Hu et al. (2024), *MiniCPM* (the WSD learning-rate schedule): [arXiv:2404.06395](https://arxiv.org/abs/2404.06395).
- Han et al. (2025), *DiariZen*: [arXiv:2409.09408](https://arxiv.org/abs/2409.09408).
- Radford et al. (2022), *Whisper*: [arXiv:2212.04356](https://arxiv.org/abs/2212.04356).
- Reddy, Gopal & Cutler (2022), *DNSMOS P.835*: [arXiv:2110.01763](https://arxiv.org/abs/2110.01763).
- Free Spoken Digit Dataset: [github.com/Jakobovski/free-spoken-digit-dataset](https://github.com/Jakobovski/free-spoken-digit-dataset).

**Back to:** [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/) ·
[Topic 05 · Audio tokenization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/token-hoa-am-thanh/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)